# **MAIN PROGRAM: IndoBERT Compression Research**

## **FASE 2: FINE TUNE BASELINE**

### **Dependency Instalation**

Install dependensi yang dibutuhkan

In [1]:
!pip uninstall -y numpy transformers peft
!pip install "numpy<2.1" "transformers>=4.41.0" datasets evaluate peft --quiet
!pip install scikit-learn pandas --quiet

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: transformers 5.9.0
Uninstalling transformers-5.9.0:
  Successfully uninstalled transformers-5.9.0
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1


In [2]:
!pip install sentencepiece tiktoken

Cek env yang digunakan

In [3]:
try:
    import os, time, json
    import numpy as np
    import torch
    from transformers import (
        AutoTokenizer, AutoModelForSequenceClassification,
        TrainingArguments, Trainer, DataCollatorWithPadding
    )
    from datasets import load_from_disk
    import evaluate

    print("IndoBERT Compression Fase 2: Baseline Fine-tuning")
    USE_GPU = torch.cuda.is_available()
    if USE_GPU:
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("GPU tidak tersedia, menggunakan CPU")
    print(f"PyTorch: {torch.__version__}")
    print(f"Numpy: {np.__version__}")
except ImportError as e:
    print(f"Import failed: {e}. Please ensure you have restarted the runtime.")

IndoBERT Compression Fase 2: Baseline Fine-tuning
GPU: Tesla T4
VRAM: 15.6 GB
PyTorch: 2.10.0+cu128
Numpy: 2.0.2


### **Dataset Loading**

In [4]:
# Konfigurasi path
DRIVE_ROOT   = "/content/drive/MyDrive/DatasetScraping_GooglePlaystore"
HF_DATA_PATH = f"{DRIVE_ROOT}/hf_dataset"
OUTPUT_DIR   = f"{DRIVE_ROOT}/indobert-baseline"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load dataset
print("Loading IndoSentiment-PlayStore dataset...")
dataset = load_from_disk(HF_DATA_PATH)

print(dataset)
print(f"\nLabel mapping: {dataset['train'].features['label'].names}")
print(f"\nContoh data training:")
for i in range(2):
    row = dataset['train'][i]
    label_name = dataset['train'].features['label'].names[row['label']]
    print(f"[{label_name}] {row['text'][:80]}...")

Loading IndoSentiment-PlayStore dataset...
DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'app_name'],
        num_rows: 8232
    })
    validation: Dataset({
        features: ['text', 'label', 'app_name'],
        num_rows: 1992
    })
    test: Dataset({
        features: ['text', 'label', 'app_name'],
        num_rows: 1993
    })
})

Label mapping: ['negatif', 'netral', 'positif']

Contoh data training:
[positif] ayo bangkit bukalapak.. dulu sering belanja disini, sekarang pas buka aplikasiny...
[negatif] lama nya bener2 kebangetan kalo mesen go food driver nya ga ketemu2...


Dataset: IndoSentiment-PlayStore (custom scraping)

### **Dataset Tokenization**

In [5]:
MODEL_NAME = "indobenchmark/indobert-base-p1"
print(f"Loading tokenizer: {MODEL_NAME}")

# Menambahkan use_fast=False untuk menghindari error sentencepiece/tiktoken
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def preprocess(examples):
    """Tokenize teks dengan max_length=128"""
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding=False
    )

print("Tokenizing dataset...")
tokenized = dataset.map(
    preprocess,
    batched=True,
    remove_columns=["text", "app_name"],
    desc="Tokenizing"
)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Verifikasi
print(f"\nTokenized features: {list(tokenized['train'].features.keys())}")
print(f"Sample: input_ids length = {len(tokenized['train'][0]['input_ids'])}")
print(f"\nSplit sizes:")
for split in ["train", "validation", "test"]:
    print(f"{split:12s}: {len(tokenized[split]):,} samples")

Loading tokenizer: indobenchmark/indobert-base-p1


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.txt: 0.00B [00:00, ?B/s]

Tokenizing dataset...


Tokenizing:   0%|          | 0/8232 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/1992 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/1993 [00:00<?, ? examples/s]


Tokenized features: ['label', 'input_ids', 'token_type_ids', 'attention_mask']
Sample: input_ids length = 41

Split sizes:
train       : 8,232 samples
validation  : 1,992 samples
test        : 1,993 samples


### **Model Initialization**

In [6]:
# Label configuration
# Sesuai dengan hasil labeling: negatif=0, netral=1, positif=2
id2label = {0: "negatif", 1: "netral", 2: "positif"}
label2id = {"negatif": 0, "netral": 1, "positif": 2}

print(f"Loading model: {MODEL_NAME}")
print(f"Num labels: 3 ({list(label2id.keys())})")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True   # classifier head baru
)

# Info model
total_params     = sum(p.numel() for p in model.parameters()) / 1e6
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"\nTotal params    : {total_params:.1f}M")
print(f"Trainable params: {trainable_params:.1f}M")
print("\nWarning 'Some weights not initialized' di atas adalah NORMAL")
print("(classifier head baru, akan ditraining dari awal)")

Loading model: indobenchmark/indobert-base-p1
Num labels: 3 (['negatif', 'netral', 'positif'])


[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `5`.


pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Total params    : 124.4M
Trainable params: 124.4M

Warning 'Some weights not initialized' di atas adalah NORMAL
(classifier head baru, akan ditraining dari awal)


### **Model Fine-Tuning**

In [7]:
# Metrics
metric_f1  = evaluate.load("f1")
metric_acc = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = metric_f1.compute(
        predictions=predictions,
        references=labels,
        average="macro"    # macro karena evaluasi per-kelas penting
    )
    acc = metric_acc.compute(predictions=predictions, references=labels)
    return {"f1_macro": f1["f1"], "accuracy": acc["accuracy"]}

# Training arguments
# Dioptimalkan untuk Colab T4 GPU (16GB VRAM)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Epoch & batch
    num_train_epochs=3,
    per_device_train_batch_size=32 if USE_GPU else 8,
    per_device_eval_batch_size=64 if USE_GPU else 16,
    gradient_accumulation_steps=1 if USE_GPU else 4,

    # Optimizer
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,

    # Evaluasi & checkpoint
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    # Efisiensi
    fp16=USE_GPU,              # Mixed precision hanya dengan GPU
    dataloader_num_workers=2,
    logging_steps=50,
    report_to="none",
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Estimasi waktu
steps_per_epoch = len(tokenized["train"]) // training_args.per_device_train_batch_size
total_steps = steps_per_epoch * training_args.num_train_epochs

print(f"\n{'='*55}")
print(f"MULAI FINE-TUNING")
print(f"{'='*55}")
print(f"Device        : {'GPU (' + torch.cuda.get_device_name(0) + ')' if USE_GPU else 'CPU'}")
print(f"Batch size    : {training_args.per_device_train_batch_size} per device")
print(f"Total steps   : {total_steps:,} ({steps_per_epoch} per epoch x 3 epochs)")
print(f"Estimasi waktu: {'15-25 menit (GPU)' if USE_GPU else '2-4 jam (CPU)'}")
print(f"{'='*55}\n")

trainer.train()
print("\nFine-tuning selesai!")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



MULAI FINE-TUNING
Device        : GPU (Tesla T4)
Batch size    : 32 per device
Total steps   : 771 (257 per epoch x 3 epochs)
Estimasi waktu: 15-25 menit (GPU)



Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy
1,0.395325,0.402973,0.817263,0.820281
2,0.318655,0.413986,0.820765,0.823795
3,0.230634,0.455943,0.822284,0.824799


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Fine-tuning selesai!


### **Evaluation**

In [9]:
print("EVALUASI PADA TEST SET")

test_results = trainer.evaluate(tokenized["test"])

print(f"\nHASIL EVALUASI:")
print(f"F1 Macro  : {test_results['eval_f1_macro']:.4f}")
print(f"Accuracy  : {test_results['eval_accuracy']:.4f}")
print(f"Loss      : {test_results['eval_loss']:.4f}")

# Evaluasi per-kelas (confusion matrix style)
from sklearn.metrics import classification_report

# Ambil predictions
preds_output = trainer.predict(tokenized["test"])
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids
label_names = ["negatif", "positif"]

print(f"\nCLASSIFICATION REPORT:")
print(classification_report(y_true, y_pred, target_names=label_names, digits=4))

# Simpan model final
final_model_path = f"{OUTPUT_DIR}/final"
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f"\nModel tersimpan: {final_model_path}")

EVALUASI PADA TEST SET


Training Loss,Validation Loss,Epoch,F1 Macro,Accuracy
0.230634,0.481947,3,0.807888,0.810336



HASIL EVALUASI:
F1 Macro  : 0.8079
Accuracy  : 0.8103
Loss      : 0.4819



CLASSIFICATION REPORT:
              precision    recall  f1-score   support

     negatif     0.8311    0.8281    0.8296      1111
     positif     0.7844    0.7880    0.7862       882

    accuracy                         0.8103      1993
   macro avg     0.8077    0.8080    0.8079      1993
weighted avg     0.8104    0.8103    0.8104      1993



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model tersimpan: /content/drive/MyDrive/DatasetScraping_GooglePlaystore/indobert-baseline/final


In [10]:
# Muat model ke CPU untuk benchmark latency
print("Loading model ke CPU untuk benchmark latency...")
model_cpu = AutoModelForSequenceClassification.from_pretrained(final_model_path)
model_cpu.eval()

def get_model_size_mb(model):
    torch.save(model.state_dict(), "/tmp/tmp_model.pt")
    size = os.path.getsize("/tmp/tmp_model.pt") / 1e6
    os.remove("/tmp/tmp_model.pt")
    return round(size, 1)

def measure_cpu_latency(model, tokenizer, n=50):
    """Ukur latency rata-rata pada beberapa jenis teks ulasan."""
    test_texts = [
        "aplikasi ini sangat bagus dan mudah digunakan sehari-hari",
        "sangat mengecewakan, sering crash dan loading lama sekali",
        "lumayan bagus tapi masih ada beberapa bug yang perlu diperbaiki"
    ]
    latencies = []
    for text in test_texts:
        inputs = tokenizer(
            text, return_tensors="pt",
            max_length=128, truncation=True
        )
        # Warmup
        with torch.no_grad():
            for _ in range(10):
                model(**inputs)
        # Measure
        start = time.perf_counter()
        with torch.no_grad():
            for _ in range(n):
                model(**inputs)
        elapsed = (time.perf_counter() - start) / n * 1000
        latencies.append(elapsed)
    return round(np.mean(latencies), 2), round(np.std(latencies), 2)

# Record semua baseline numbers
model_size = get_model_size_mb(model_cpu)
num_params = sum(p.numel() for p in model_cpu.parameters()) / 1e6

print("Mengukur CPU latency (ini agak lama ~2 menit)...")
lat_mean, lat_std = measure_cpu_latency(model_cpu, tokenizer)

# Print tabel hasil
print("\n" + "="*60)
print("BASELINE NUMBERS - BARIS PERTAMA TABEL ABLATION STUDY")
print("="*60)
print(f"  Model          : IndoBERT-base (no compression)")
print(f"  Dataset        : IndoSentiment-PlayStore (custom)")
print("\n" + "="*60)
print(f"  Parameters     : {num_params:.1f}M")
print(f"  Model Size     : {model_size} MB")
print(f"  F1 Macro (test): {test_results['eval_f1_macro']:.4f}")
print(f"  Accuracy (test): {test_results['eval_accuracy']:.4f}")
print(f"  CPU Latency    : {lat_mean} ± {lat_std} ms/sample")
print(f"  Speedup        : 1× (baseline)")
print("="*60)
print("\n  Catat angka-angka ini — ini adalah TARGET kompresi!")
print(f"   Goal: Size < 30MB (~{model_size/30:.0f}× lebih kecil)")
print(f"   Goal: Latency < 20ms (~{lat_mean/20:.0f}× lebih cepat)")

# Simpan JSON untuk referensi eksperimen berikutnya
baseline_data = {
    "experiment": "baseline_no_compression",
    "model": "IndoBERT-base",
    "dataset": "IndoSentiment-PlayStore",
    "pruning": False,
    "distillation": False,
    "quantization": False,
    "onnx": False,
    "num_parameters_M": round(num_params, 1),
    "model_size_mb": model_size,
    "f1_macro": round(test_results['eval_f1_macro'], 4),
    "accuracy": round(test_results['eval_accuracy'], 4),
    "eval_loss": round(test_results['eval_loss'], 4),
    "cpu_latency_ms": lat_mean,
    "cpu_latency_std_ms": lat_std,
    "speedup_vs_baseline": 1.0
}

results_path = f"{OUTPUT_DIR}/baseline_results.json"
with open(results_path, "w") as f:
    json.dump(baseline_data, f, indent=2, ensure_ascii=False)

print(f"\n Semua data tersimpan di Drive:")
print(f"   Model   → {final_model_path}")
print(f"   Results → {results_path}")

Loading model ke CPU untuk benchmark latency...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Mengukur CPU latency (ini agak lama ~2 menit)...

BASELINE NUMBERS - BARIS PERTAMA TABEL ABLATION STUDY
  Model          : IndoBERT-base (no compression)
  Dataset        : IndoSentiment-PlayStore (custom)

  Parameters     : 124.4M
  Model Size     : 497.9 MB
  F1 Macro (test): 0.8079
  Accuracy (test): 0.8103
  CPU Latency    : 117.35 ± 8.93 ms/sample
  Speedup        : 1× (baseline)

  Catat angka-angka ini — ini adalah TARGET kompresi!
   Goal: Size < 30MB (~17× lebih kecil)
   Goal: Latency < 20ms (~6× lebih cepat)

 Semua data tersimpan di Drive:
   Model   → /content/drive/MyDrive/DatasetScraping_GooglePlaystore/indobert-baseline/final
   Results → /content/drive/MyDrive/DatasetScraping_GooglePlaystore/indobert-baseline/baseline_results.json
